In [5]:
import sys
sys.path.append('..')

import pandas as pd 
import numpy as np 
from scipy import stats
import matplotlib.pyplot as plt 
import seaborn as sns 

from src.data_loader import DataLoader 
from src.statistics import StatisticalAnalyzer 

# Load cleaned data 
loader = DataLoader() 
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv") 

# Initialize analyzer 
analyzer = StatisticalAnalyzer(df) 

print("=" * 60) 
print("HYPOTHESIS TESTING") 
print("=" * 60) 

all_tests = []

# ==================== 1. T-TESTS ==================== 
print("\n T-TESTS") 

# One-sample t-test: Is average game price significantly different from $20.00?
if 'price' in df.columns: 
    result = analyzer.t_test('price', 20.0) 
    print(f"\nOne-sample t-test on price (testing against $20.00):") 
    print(f"  Mean: ${result['mean']:.2f}") 
    print(f"  t-statistic: {result['statistic']:.3f}") 
    print(f"  p-value: {result['p_value']:.4f}") 
    print(f"  Significant: {result['significant']}") 
    print(f"  Interpretation: {result['interpretation']}") 
    
    all_tests.append({
        'test': result['test'],
        'variable1': result['variable'],
        'variable2': f"mu={result['null_value']}",
        'statistic': result['statistic'],
        'p_value': result['p_value'],
        'significant': result['significant']
    })

# Independent t-test: Compare reviewScore between top two publisherClass groups
if 'reviewScore' in df.columns and 'publisherClass' in df.columns: 
    pub_classes = df['publisherClass'].dropna().unique() 
    if len(pub_classes) >= 2: 
        group1 = pub_classes[0] 
        group2 = pub_classes[1] 
         
        data1 = df[df['publisherClass'] == group1]['reviewScore'].dropna() 
        data2 = df[df['publisherClass'] == group2]['reviewScore'].dropna() 
         
        statistic, p_value = stats.ttest_ind(data1, data2) 
         
        print(f"\nIndependent t-test on reviewScore: {group1} vs {group2}") 
        print(f"  Mean {group1}: {data1.mean():.3f}") 
        print(f"  Mean {group2}: {data2.mean():.3f}") 
        print(f"  t-statistic: {statistic:.3f}") 
        print(f"  p-value: {p_value:.4f}") 
        print(f"  Significant: {p_value < 0.05}") 
        
        all_tests.append({
            'test': 'Independent t-test',
            'variable1': f"reviewScore ({group1})",
            'variable2': f"reviewScore ({group2})",
            'statistic': statistic,
            'p_value': p_value,
            'significant': p_value < 0.05
        })

# ==================== 2. ANOVA ==================== 
print("\n ONE-WAY ANOVA") 

# ANOVA: Compare reviewScore across all publisherClass tiers
if 'reviewScore' in df.columns and 'publisherClass' in df.columns: 
    groups = [] 
    for pub_cls in df['publisherClass'].dropna().unique(): 
        group_data = df[df['publisherClass'] == pub_cls]['reviewScore'].dropna()
        if len(group_data) > 0:
            groups.append(group_data) 
     
    if len(groups) >= 2:
        f_statistic, p_value = stats.f_oneway(*groups) 
         
        print(f"\nANOVA: reviewScore by publisherClass") 
        print(f"  f-statistic: {f_statistic:.3f}") 
        print(f"  p-value: {p_value:.4f}") 
        print(f"  Significant: {p_value < 0.05}") 
         
        if p_value < 0.05: 
            print("  Interpretation: At least one publisher class has significantly different review scores") 
            
        all_tests.append({
            'test': 'One-way ANOVA',
            'variable1': 'reviewScore',
            'variable2': 'publisherClass',
            'statistic': f_statistic,
            'p_value': p_value,
            'significant': p_value < 0.05
        })

# ==================== 3. CHI-SQUARE TEST ==================== 
print("\n CHI-SQUARE TEST") 

# Chi-Square Test of Independence between publisherClass and publishers (or developers)
if 'publisherClass' in df.columns and 'publishers' in df.columns: 
    contingency = pd.crosstab(df['publisherClass'], df['publishers']) 
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency) 
     
    print(f"\nChi-square Test: publisherClass vs publishers") 
    print(f"  chi2-statistic: {chi2:.3f}") 
    print(f"  p-value: {p_value:.4f}") 
    print(f"  degrees of freedom: {dof}") 
    print(f"  Significant: {p_value < 0.05}") 
    print(f"  Interpretation: {'Variables are dependent' if p_value < 0.05 else 'Variables are independent'}") 
    
    all_tests.append({
        'test': 'Chi-square Test of Independence',
        'variable1': 'publisherClass',
        'variable2': 'publishers',
        'statistic': chi2,
        'p_value': p_value,
        'significant': p_value < 0.05
    })

# ==================== 4. SUMMARY OF ALL TESTS ==================== 
print("\n" + "=" * 60) 
print("SUMMARY OF HYPOTHESIS TESTS") 
print("=" * 60) 

summary_df = pd.DataFrame(all_tests) 
if not summary_df.empty: 
    summary_df.to_csv('../reports/hypothesis_tests_summary.csv', index=False) 
    print("\n Hypothesis test results saved to 'reports/hypothesis_tests_summary.csv'")

HYPOTHESIS TESTING

 T-TESTS

One-sample t-test on price (testing against $20.00):
  Mean: $17.52
  t-statistic: -7.596
  p-value: 0.0000
  Significant: True
  Interpretation: Mean significantly differs from 20.0

Independent t-test on reviewScore: AAA vs Indie
  Mean AAA: 72.750
  Mean Indie: 76.566
  t-statistic: -1.117
  p-value: 0.2642
  Significant: False

 ONE-WAY ANOVA

ANOVA: reviewScore by publisherClass
  f-statistic: 1.176
  p-value: 0.3089
  Significant: False

 CHI-SQUARE TEST

Chi-square Test: publisherClass vs publishers
  chi2-statistic: 3000.000
  p-value: 0.0000
  degrees of freedom: 2260
  Significant: True
  Interpretation: Variables are dependent

SUMMARY OF HYPOTHESIS TESTS

 Hypothesis test results saved to 'reports/hypothesis_tests_summary.csv'
